In [ ]:
!pip install opencv-python-headless gradio pillow numpy -q

print("✅ Basic dependencies installed")

✅ Basic dependencies installed


In [ ]:
import cv2
import numpy as np
from PIL import Image
from skimage import exposure

def enhance_image(input_image):


    img = np.array(input_image)


    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)


    denoised = cv2.bilateralFilter(img_bgr, 9, 75, 75)


    lab = cv2.cvtColor(denoised, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    l_clahe = clahe.apply(l)

    lab_enhanced = cv2.merge([l_clahe, a, b])
    enhanced = cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2BGR)


    kernel = np.array([[-1,-1,-1],
                       [-1, 9,-1],
                       [-1,-1,-1]])
    sharpened = cv2.filter2D(enhanced, -1, kernel)


    h, w = sharpened.shape[:2]
    upscaled = cv2.resize(sharpened, (w*2, h*2), interpolation=cv2.INTER_CUBIC)


    blurred = cv2.GaussianBlur(upscaled, (0, 0), 1.0)
    unsharp = cv2.addWeighted(upscaled, 1.5, blurred, -0.5, 0)


    result = cv2.cvtColor(unsharp, cv2.COLOR_BGR2RGB)

    return Image.fromarray(result.astype('uint8'))

print("✅ Enhancement function ready")

✅ Enhancement function ready


In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=enhance_image,
    inputs=gr.Image(type="pil", label="📸 Upload your photo"),
    outputs=gr.Image(label="✨ Enhanced Result"),
    title="🖼️ AI Image Enhancer (OpenCV)",
    description="Enhance photo quality with advanced OpenCV algorithms:\n- Noise reduction (Bilateral Filter)\n- Contrast enhancement (CLAHE)\n- Sharpening (Unsharp Mask)\n- 2x Upscaling",
    examples=None
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2c06fd50ff0a3c98bc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
